# 04 — Alberta AER Carbon Sequestration Agreements

## Objective

Construct an exploratory GeoPackage from the Alberta Energy Regulator (AER) carbon sequestration agreement dataset.

This notebook is intended to formalize the structure of the AER agreement data before the workflow is converted into reusable project code.

The main goals are to:

- inspect and preserve the source agreement structure;
- standardize field names and data types;
- reproject spatial data to the project working CRS, EPSG:3347;
- distinguish agreement-level records from agreement-tract records;
- retain source-reported attributes separately from derived spatial attributes;
- add provenance and quality-control fields;
- validate geometry and identifiers;
- export a clean exploratory GeoPackage for later comparison and integration with other geological storage datasets.

## Scope

This notebook treats the AER dataset as a **regulatory / tenure dataset**.

It does **not** infer:

- geological storage capacity;
- injectivity;
- reservoir quality;
- storage probability;
- technically recoverable storage resource;
- project feasibility.

Those geological attributes will be handled separately when the AER agreement layer is linked with geological datasets such as NATCARB, provincial storage atlases, or other storage-resource sources.

## Intended output

The exploratory GeoPackage will contain, at minimum:

1. **Agreement-tract layer**  
   Preserves the source spatial records and tract structure.

2. **Agreement layer**  
   Dissolves tract geometries to one record per agreement where appropriate.

3. **Supporting metadata / QA information**  
   Records provenance, transformations, validation results, and relevant field definitions.

The final structure developed here will later inform the production implementation under `src/` and the associated bronze acquisition workflow.

In [1]:
import geopandas as gpd
import pandas as pd
import sqlite3

from pathlib import Path

In [2]:
# ---------------------------------------------------------------------------
# Define source and output directories
# ---------------------------------------------------------------------------

AER_DIR = Path(
    r"C:\Users\aviga\Research\potential data\Storage\AER\CS_agreements"
)

OUTPUT_DIR = Path(
    r"C:\Users\aviga\Research\potential data\Storage\AER\derived"
)

print("AER directory exists:", AER_DIR.exists())
print("Output directory exists:", OUTPUT_DIR.exists())

AER directory exists: True
Output directory exists: True


In [3]:
# ---------------------------------------------------------------------------
# Load AER carbon sequestration agreement shapefile
# ---------------------------------------------------------------------------

shapefiles = sorted(AER_DIR.glob("*.shp"))

if len(shapefiles) != 1:
    raise ValueError(
        f"Expected exactly one shapefile in {AER_DIR}, "
        f"found {len(shapefiles)}: {[path.name for path in shapefiles]}"
    )

AER_SHP = shapefiles[0]

gdf = gpd.read_file(AER_SHP)

print("Loaded:", AER_SHP.name)
print("Features:", len(gdf))
print("CRS:", gdf.crs)

Loaded: CS_Agreements.shp
Features: 45
CRS: EPSG:3400


In [4]:
# ---------------------------------------------------------------------------
# Inspect source structure
# ---------------------------------------------------------------------------

print("Shape:", gdf.shape)
print("\nColumns:")
print(gdf.columns.tolist())

print("\nData types:")
display(gdf.dtypes.to_frame("dtype"))

print("\nFirst rows:")
display(gdf.head())

Shape: (45, 17)

Columns:
['AgreementT', 'AgreementN', 'Tract', 'MINTYPE', 'AGGROUP', 'Status', 'Vintage', 'DesRep', 'ZoneDesc', 'ORGAREA', 'AGREEAREA', 'TERMDATE', 'CONTDATE', 'CUREXPIRY', 'Shape_STAr', 'Shape_STLe', 'geometry']

Data types:


,dtype
AgreementT,str
AgreementN,str
Tract,str
MINTYPE,str
AGGROUP,str
Status,str
Vintage,str
DesRep,str
ZoneDesc,str
ORGAREA,float64



First rows:


,AgreementT,AgreementN,Tract,MINTYPE,AGGROUP,Status,Vintage,DesRep,ZoneDesc,ORGAREA,AGREEAREA,TERMDATE,CONTDATE,CUREXPIRY,Shape_STAr,Shape_STLe,geometry
0,059,5911050001,00,PORE SPACE,LEASE,ACTIVE,INITIAL TERM,SHELL CANADA LIMITED,PORE SPACE BELOW THE TOP OF THE ELK POINT GRP(...,73728.0,73728.0,2011/05/27,None,2029/05/27,7.449617e+08,1.388939e+06,"MULTIPOLYGON (((640936.64 5990467.876, 640603...."
1,059,5911050002,00,PORE SPACE,LEASE,ACTIVE,INITIAL TERM,SHELL CANADA LIMITED,PORE SPACE BELOW THE TOP OF THE ELK POINT GRP(...,64512.0,64512.0,2011/05/27,None,2029/05/27,6.498513e+08,1.213404e+06,"MULTIPOLYGON (((633666.916 5961193.292, 633264..."
2,059,5911050003,00,PORE SPACE,LEASE,ACTIVE,INITIAL TERM,SHELL CANADA LIMITED,PORE SPACE BELOW THE TOP OF THE ELK POINT GRP(...,64512.0,64512.0,2011/05/27,None,2029/05/27,6.495719e+08,1.213292e+06,"MULTIPOLYGON (((621144.545 5999654.437, 620742..."
3,059,5911050004,00,PORE SPACE,LEASE,ACTIVE,INITIAL TERM,SHELL CANADA LIMITED,PORE SPACE BELOW THE TOP OF THE ELK POINT GRP(...,49152.0,49152.0,2011/05/27,None,2029/05/27,4.943340e+08,9.240810e+05,"MULTIPOLYGON (((623703.7 5967399.859, 623300.8..."
4,059,5911050005,00,PORE SPACE,LEASE,ACTIVE,INITIAL TERM,SHELL CANADA LIMITED,PORE SPACE BELOW THE TOP OF THE ELK POINT GRP(...,55296.0,55296.0,2011/05/27,None,2029/05/27,5.583449e+08,1.041488e+06,"MULTIPOLYGON (((603862.508 5979868.628, 603461..."


In [5]:
# ---------------------------------------------------------------------------
# Inspect nulls and unique values
# ---------------------------------------------------------------------------

field_summary = pd.DataFrame(
    {
        "dtype": gdf.dtypes.astype(str),
        "null_count": gdf.isna().sum(),
        "non_null_count": gdf.notna().sum(),
        "unique_count": gdf.nunique(dropna=True),
    }
)

display(field_summary)

,dtype,null_count,non_null_count,unique_count
AgreementT,str,0,45,6
AgreementN,str,0,45,43
Tract,str,0,45,3
MINTYPE,str,0,45,1
AGGROUP,str,0,45,3
Status,str,0,45,1
Vintage,str,30,15,1
DesRep,str,0,45,25
ZoneDesc,str,0,45,25
ORGAREA,float64,0,45,36


In [6]:
# ---------------------------------------------------------------------------
# Inspect categorical field values
# ---------------------------------------------------------------------------

categorical_fields = [
    "AgreementT",
    "Tract",
    "MINTYPE",
    "AGGROUP",
    "Status",
    "Vintage",
]

for field in categorical_fields:
    print(f"\n{field}")
    print("-" * len(field))
    print(gdf[field].value_counts(dropna=False))


AgreementT
----------
AgreementT
058    16
059    15
061     7
A61     4
A59     2
A58     1
Name: count, dtype: int64

Tract
-----
Tract
00    41
01     2
02     2
Name: count, dtype: int64

MINTYPE
-------
MINTYPE
PORE SPACE    45
Name: count, dtype: int64

AGGROUP
-------
AGGROUP
LEASE     22
PERMIT    16
APP        7
Name: count, dtype: int64

Status
------
Status
ACTIVE    45
Name: count, dtype: int64

Vintage
-------
Vintage
NaN             30
INITIAL TERM    15
Name: count, dtype: int64


In [7]:
# ---------------------------------------------------------------------------
# Inspect agreement identifiers, representatives, zones, and dates
# ---------------------------------------------------------------------------

display(
    gdf[
        [
            "AgreementN",
            "Tract",
            "AGGROUP",
            "AgreementT",
            "DesRep",
            "ZoneDesc",
            "TERMDATE",
            "CONTDATE",
            "CUREXPIRY",
        ]
    ]
    .sort_values(["AgreementN", "Tract"])
    .reset_index(drop=True)
)

,AgreementN,Tract,AGGROUP,AgreementT,DesRep,ZoneDesc,TERMDATE,CONTDATE,CUREXPIRY
0,240148301,00,APP,A59,ALTAGAS LTD.,PORE SPACE IN THE WOODBEND GRP(D300),2024/07/02,None,NaN
1,250085901,00,APP,A61,CANADIAN NATURAL RESOURCES LIMITED,PORE SPACE IN THE WINTERBURN GRP(D80),2025/05/29,None,NaN
2,250108401,00,APP,A61,CENTRAL FARMS RNG LTD.,PORE SPACE IN THE LIVINGSTONE FM(D236),2025/06/30,None,NaN
3,260043401,00,APP,A61,ALTAGAS LTD.,PORE SPACE BELOW THE TOP OF THE BELLOY FM(D32)...,2026/03/03,None,NaN
4,260088401,00,APP,A61,KEYERA ENERGY LTD.,PORE SPACE IN THE LEDUC FM(D67),2026/05/07,None,NaN
5,260116601,00,APP,A59,ENBRIDGE WABAMUN HUB LTD.,PORE SPACE IN THE WINTERBURN GRP,2026/06/09,None,NaN
6,260127901,00,APP,A58,VAULT 44.01 LTD.,PORE SPACE BELOW THE TOP OF THE BANFF FM(D255)...,2026/06/29,None,NaN
7,5822100007,00,PERMIT,058,ENBRIDGE WABAMUN HUB LTD.,PORE SPACE IN THE BASAL SANDSTONE UNIT(D298) P...,2022/10/01,None,2027/10/01
8,5822100008,00,PERMIT,058,ENHANCE ENERGY INC.,PORE SPACE IN THE WOODBEND GRP,2022/10/01,None,2027/10/01
9,5822100009,00,PERMIT,058,BISON LOW CARBON VENTURES INC.,PORE SPACE IN THE WOODBEND GRP,2022/10/01,None,2027/10/01


In [8]:
# ---------------------------------------------------------------------------
# Inspect multi-tract agreements
# ---------------------------------------------------------------------------

multi_tract_ids = (
    gdf.groupby("AgreementN")
    .size()
    .loc[lambda s: s > 1]
    .index
)

multi_tract = (
    gdf[gdf["AgreementN"].isin(multi_tract_ids)]
    [
        [
            "AgreementN",
            "Tract",
            "AGGROUP",
            "AgreementT",
            "DesRep",
            "ZoneDesc",
            "ORGAREA",
            "AGREEAREA",
        ]
    ]
    .sort_values(["AgreementN", "Tract"])
    .reset_index(drop=True)
)

display(multi_tract)

,AgreementN,Tract,AGGROUP,AgreementT,DesRep,ZoneDesc,ORGAREA,AGREEAREA
0,5924090003,01,LEASE,059,ENHANCE ENERGY INC.,PORE SPACE IN THE WOODBEND GRP,483072.0,483072.0
1,5924090003,02,LEASE,059,ENHANCE ENERGY INC.,EXCEPTING PORE SPACE IN THE LEDUC FM(D7) PORE ...,483072.0,483072.0
2,5924090004,01,LEASE,059,WOLF CARBON HUB GP INC.,PORE SPACE BELOW THE TOP OF THE EARLIE FM(D302...,184064.0,184064.0
3,5924090004,02,LEASE,059,WOLF CARBON HUB GP INC.,PORE SPACE IN THE BASAL SANDSTONE UNIT(D298),184064.0,184064.0


In [9]:
# ---------------------------------------------------------------------------
# Validate agreement-level attribute consistency across tracts
# ---------------------------------------------------------------------------

agreement_level_fields = [
    "AGGROUP",
    "AgreementT",
    "DesRep",
    "ZoneDesc",
    "ORGAREA",
    "AGREEAREA",
    "TERMDATE",
    "CONTDATE",
    "CUREXPIRY",
]

consistency = (
    gdf.groupby("AgreementN")[agreement_level_fields]
    .nunique(dropna=False)
)

inconsistent = consistency[
    (consistency > 1).any(axis=1)
]

if inconsistent.empty:
    print("All agreement-level attributes are consistent across multi-tract agreements.")
else:
    print("Inconsistent agreement-level attributes detected:")
    display(inconsistent)

Inconsistent agreement-level attributes detected:


,AGGROUP,AgreementT,DesRep,ZoneDesc,ORGAREA,AGREEAREA,TERMDATE,CONTDATE,CUREXPIRY
AgreementN,,,,,,,,,
5924090003,1,1,1,2,1,1,1,1,1
5924090004,1,1,1,2,1,1,1,1,1


In [10]:
# ---------------------------------------------------------------------------
# Define canonical field names
# ---------------------------------------------------------------------------

rename_map = {
    "AgreementN": "agreement_id",
    "AgreementT": "agreement_type_code",
    "Tract": "tract_id",
    "MINTYPE": "mineral_type",
    "AGGROUP": "agreement_group",
    "Status": "status",
    "Vintage": "vintage",
    "DesRep": "designated_representative",
    "ZoneDesc": "zone_description",
    "ORGAREA": "original_area",
    "AGREEAREA": "agreement_area",
    "TERMDATE": "term_date",
    "CONTDATE": "continuation_date",
    "CUREXPIRY": "current_expiry",
    "Shape_STAr": "source_shape_area",
    "Shape_STLe": "source_shape_length",
}

clean_gdf = gdf.rename(columns=rename_map).copy()

display(clean_gdf.head())

,agreement_type_code,agreement_id,tract_id,mineral_type,agreement_group,status,vintage,designated_representative,zone_description,original_area,agreement_area,term_date,continuation_date,current_expiry,source_shape_area,source_shape_length,geometry
0,059,5911050001,00,PORE SPACE,LEASE,ACTIVE,INITIAL TERM,SHELL CANADA LIMITED,PORE SPACE BELOW THE TOP OF THE ELK POINT GRP(...,73728.0,73728.0,2011/05/27,None,2029/05/27,7.449617e+08,1.388939e+06,"MULTIPOLYGON (((640936.64 5990467.876, 640603...."
1,059,5911050002,00,PORE SPACE,LEASE,ACTIVE,INITIAL TERM,SHELL CANADA LIMITED,PORE SPACE BELOW THE TOP OF THE ELK POINT GRP(...,64512.0,64512.0,2011/05/27,None,2029/05/27,6.498513e+08,1.213404e+06,"MULTIPOLYGON (((633666.916 5961193.292, 633264..."
2,059,5911050003,00,PORE SPACE,LEASE,ACTIVE,INITIAL TERM,SHELL CANADA LIMITED,PORE SPACE BELOW THE TOP OF THE ELK POINT GRP(...,64512.0,64512.0,2011/05/27,None,2029/05/27,6.495719e+08,1.213292e+06,"MULTIPOLYGON (((621144.545 5999654.437, 620742..."
3,059,5911050004,00,PORE SPACE,LEASE,ACTIVE,INITIAL TERM,SHELL CANADA LIMITED,PORE SPACE BELOW THE TOP OF THE ELK POINT GRP(...,49152.0,49152.0,2011/05/27,None,2029/05/27,4.943340e+08,9.240810e+05,"MULTIPOLYGON (((623703.7 5967399.859, 623300.8..."
4,059,5911050005,00,PORE SPACE,LEASE,ACTIVE,INITIAL TERM,SHELL CANADA LIMITED,PORE SPACE BELOW THE TOP OF THE ELK POINT GRP(...,55296.0,55296.0,2011/05/27,None,2029/05/27,5.583449e+08,1.041488e+06,"MULTIPOLYGON (((603862.508 5979868.628, 603461..."


In [11]:
# ---------------------------------------------------------------------------
# Standardize field data types
# ---------------------------------------------------------------------------

# Preserve agreement and tract identifiers as strings
clean_gdf["agreement_id"] = clean_gdf["agreement_id"].astype("string")
clean_gdf["tract_id"] = clean_gdf["tract_id"].astype("string")
clean_gdf["agreement_type_code"] = clean_gdf["agreement_type_code"].astype("string")

# Convert date fields
date_fields = [
    "term_date",
    "continuation_date",
    "current_expiry",
]

for field in date_fields:
    clean_gdf[field] = pd.to_datetime(
        clean_gdf[field],
        errors="coerce",
    )

# Confirm resulting data types
display(clean_gdf.dtypes.to_frame("dtype"))

,dtype
agreement_type_code,string
agreement_id,string
tract_id,string
mineral_type,str
agreement_group,str
status,str
vintage,str
designated_representative,str
zone_description,str
original_area,float64


In [12]:
# ---------------------------------------------------------------------------
# Validate geometry and reproject to CanCO2Re working CRS
# ---------------------------------------------------------------------------

print("Source CRS:", clean_gdf.crs)
print("Null geometries:", clean_gdf.geometry.isna().sum())
print("Empty geometries:", clean_gdf.geometry.is_empty.sum())
print("Invalid geometries:", (~clean_gdf.geometry.is_valid).sum())

TARGET_CRS = "EPSG:3978"

clean_gdf = clean_gdf.to_crs(TARGET_CRS)

print("\nReprojected CRS:", clean_gdf.crs)

Source CRS: EPSG:3400
Null geometries: 0
Empty geometries: 0
Invalid geometries: 0

Reprojected CRS: EPSG:3978


In [13]:
# ---------------------------------------------------------------------------
# Compare source and derived geometry measurements
# ---------------------------------------------------------------------------

measurement_check = clean_gdf[
    [
        "agreement_id",
        "tract_id",
        "original_area",
        "agreement_area",
        "source_shape_area",
        "source_shape_length",
    ]
].copy()

measurement_check["derived_area_m2"] = clean_gdf.geometry.area
measurement_check["derived_length_m"] = clean_gdf.geometry.length

measurement_check["shape_area_ratio"] = (
    measurement_check["source_shape_area"]
    / measurement_check["derived_area_m2"]
)

measurement_check["shape_length_ratio"] = (
    measurement_check["source_shape_length"]
    / measurement_check["derived_length_m"]
)

display(measurement_check.head(10))

print("\nMedian source/derived area ratio:")
print(measurement_check["shape_area_ratio"].median())

print("\nMedian source/derived length ratio:")
print(measurement_check["shape_length_ratio"].median())

,agreement_id,tract_id,original_area,agreement_area,source_shape_area,source_shape_length,derived_area_m2,derived_length_m,shape_area_ratio,shape_length_ratio
0,5911050001,00,73728.0,73728.000,7.449617e+08,1.388939e+06,7.206913e+08,1.366126e+06,1.033677,1.016699
1,5911050002,00,64512.0,64512.000,6.498513e+08,1.213404e+06,6.295819e+08,1.194330e+06,1.032195,1.015970
2,5911050003,00,64512.0,64512.000,6.495719e+08,1.213292e+06,6.280638e+08,1.193036e+06,1.034245,1.016978
3,5911050004,00,49152.0,49152.000,4.943340e+08,9.240810e+05,4.788737e+08,9.095157e+05,1.032285,1.016014
4,5911050005,00,55296.0,55296.000,5.583449e+08,1.041488e+06,5.405159e+08,1.024725e+06,1.032985,1.016359
5,5911050006,00,55296.0,55296.000,5.557155e+08,1.038917e+06,5.373884e+08,1.021641e+06,1.034104,1.016909
6,5822100007,00,211072.0,140976.026,1.421949e+09,2.690536e+06,1.382420e+09,2.652858e+06,1.028594,1.014203
7,5822100008,00,544512.0,52480.000,5.301950e+08,1.003367e+06,5.191233e+08,9.928290e+05,1.021328,1.010614
8,5822100009,00,70400.0,55040.000,5.557629e+08,1.041248e+06,5.383421e+08,1.024800e+06,1.032360,1.016051
9,5822100010,00,950720.0,607996.600,6.141837e+09,1.148630e+07,5.929893e+09,1.128640e+07,1.035742,1.017712



Median source/derived area ratio:
1.031394552326586

Median source/derived length ratio:
1.0155750778203239


In [14]:
# ---------------------------------------------------------------------------
# Standardize measurement field names and units
# ---------------------------------------------------------------------------

clean_gdf = clean_gdf.rename(
    columns={
        "original_area": "original_area_ha",
        "agreement_area": "agreement_area_ha",
        "source_shape_area": "source_shape_area_m2",
        "source_shape_length": "source_shape_length_m",
    }
)

display(
    clean_gdf[
        [
            "agreement_id",
            "tract_id",
            "original_area_ha",
            "agreement_area_ha",
            "source_shape_area_m2",
            "source_shape_length_m",
        ]
    ].head()
)

,agreement_id,tract_id,original_area_ha,agreement_area_ha,source_shape_area_m2,source_shape_length_m
0,5911050001,00,73728.0,73728.0,7.449617e+08,1.388939e+06
1,5911050002,00,64512.0,64512.0,6.498513e+08,1.213404e+06
2,5911050003,00,64512.0,64512.0,6.495719e+08,1.213292e+06
3,5911050004,00,49152.0,49152.0,4.943340e+08,9.240810e+05
4,5911050005,00,55296.0,55296.0,5.583449e+08,1.041488e+06


In [15]:
# ---------------------------------------------------------------------------
# Add geometry-derived measurements in EPSG:3978
# ---------------------------------------------------------------------------

clean_gdf["geometry_area_m2"] = clean_gdf.geometry.area
clean_gdf["geometry_area_ha"] = clean_gdf["geometry_area_m2"] / 10_000
clean_gdf["geometry_perimeter_m"] = clean_gdf.geometry.length

display(
    clean_gdf[
        [
            "agreement_id",
            "tract_id",
            "agreement_area_ha",
            "geometry_area_ha",
            "source_shape_area_m2",
            "geometry_area_m2",
            "source_shape_length_m",
            "geometry_perimeter_m",
        ]
    ].head()
)

,agreement_id,tract_id,agreement_area_ha,geometry_area_ha,source_shape_area_m2,geometry_area_m2,source_shape_length_m,geometry_perimeter_m
0,5911050001,00,73728.0,72069.130239,7.449617e+08,7.206913e+08,1.388939e+06,1.366126e+06
1,5911050002,00,64512.0,62958.190706,6.498513e+08,6.295819e+08,1.213404e+06,1.194330e+06
2,5911050003,00,64512.0,62806.381531,6.495719e+08,6.280638e+08,1.213292e+06,1.193036e+06
3,5911050004,00,49152.0,47887.365632,4.943340e+08,4.788737e+08,9.240810e+05,9.095157e+05
4,5911050005,00,55296.0,54051.588175,5.583449e+08,5.405159e+08,1.041488e+06,1.024725e+06


In [16]:
# ---------------------------------------------------------------------------
# Add feature-level provenance
# ---------------------------------------------------------------------------

clean_gdf["source_dataset"] = "AER Carbon Sequestration Agreements"

clean_gdf["source_feature_id"] = clean_gdf.index.astype(int)

clean_gdf["source_feature_uid"] = (
    clean_gdf["agreement_id"].astype(str)
    + ":"
    + clean_gdf["tract_id"].astype(str)
)

clean_gdf["source_crs"] = "EPSG:3400"

display(
    clean_gdf[
        [
            "agreement_id",
            "tract_id",
            "source_dataset",
            "source_feature_id",
            "source_feature_uid",
            "source_crs",
        ]
    ].head()
)

,agreement_id,tract_id,source_dataset,source_feature_id,source_feature_uid,source_crs
0,5911050001,00,AER Carbon Sequestration Agreements,0,5911050001:00,EPSG:3400
1,5911050002,00,AER Carbon Sequestration Agreements,1,5911050002:00,EPSG:3400
2,5911050003,00,AER Carbon Sequestration Agreements,2,5911050003:00,EPSG:3400
3,5911050004,00,AER Carbon Sequestration Agreements,3,5911050004:00,EPSG:3400
4,5911050005,00,AER Carbon Sequestration Agreements,4,5911050005:00,EPSG:3400


In [17]:
# ---------------------------------------------------------------------------
# Define canonical tract-layer schema
# ---------------------------------------------------------------------------

tract_columns = [
    # Core identifiers
    "agreement_id",
    "tract_id",

    # Agreement classification
    "agreement_group",
    "agreement_type_code",
    "mineral_type",
    "status",
    "vintage",

    # Agreement holder and geological description
    "designated_representative",
    "zone_description",

    # Administrative areas and dates
    "original_area_ha",
    "agreement_area_ha",
    "term_date",
    "continuation_date",
    "current_expiry",

    # Source geometry measurements
    "source_shape_area_m2",
    "source_shape_length_m",

    # Derived geometry measurements in EPSG:3978
    "geometry_area_m2",
    "geometry_area_ha",
    "geometry_perimeter_m",

    # Feature-level provenance
    "source_dataset",
    "source_feature_id",
    "source_feature_uid",
    "source_crs",

    # Geometry
    "geometry",
]

tract_gdf = clean_gdf[tract_columns].copy()

display(tract_gdf.head())
print("\nColumns:", len(tract_gdf.columns))
print("Features:", len(tract_gdf))
print("CRS:", tract_gdf.crs)

,agreement_id,tract_id,agreement_group,agreement_type_code,mineral_type,status,vintage,designated_representative,zone_description,original_area_ha,...,source_shape_area_m2,source_shape_length_m,geometry_area_m2,geometry_area_ha,geometry_perimeter_m,source_dataset,source_feature_id,source_feature_uid,source_crs,geometry
0,5911050001,00,LEASE,059,PORE SPACE,ACTIVE,INITIAL TERM,SHELL CANADA LIMITED,PORE SPACE BELOW THE TOP OF THE ELK POINT GRP(...,73728.0,...,7.449617e+08,1.388939e+06,7.206913e+08,72069.130239,1.366126e+06,AER Carbon Sequestration Agreements,0,5911050001:00,EPSG:3400,"MULTIPOLYGON (((-1134070.696 718678.172, -1134..."
1,5911050002,00,LEASE,059,PORE SPACE,ACTIVE,INITIAL TERM,SHELL CANADA LIMITED,PORE SPACE BELOW THE TOP OF THE ELK POINT GRP(...,64512.0,...,6.498513e+08,1.213404e+06,6.295819e+08,62958.190706,1.194330e+06,AER Carbon Sequestration Agreements,1,5911050002:00,EPSG:3400,"MULTIPOLYGON (((-1149703.307 693434.615, -1150..."
2,5911050003,00,LEASE,059,PORE SPACE,ACTIVE,INITIAL TERM,SHELL CANADA LIMITED,PORE SPACE BELOW THE TOP OF THE ELK POINT GRP(...,64512.0,...,6.495719e+08,1.213292e+06,6.280638e+08,62806.381531,1.193036e+06,AER Carbon Sequestration Agreements,2,5911050003:00,EPSG:3400,"MULTIPOLYGON (((-1149840.406 733246.73, -11502..."
3,5911050004,00,LEASE,059,PORE SPACE,ACTIVE,INITIAL TERM,SHELL CANADA LIMITED,PORE SPACE BELOW THE TOP OF THE ELK POINT GRP(...,49152.0,...,4.943340e+08,9.240810e+05,4.788737e+08,47887.365632,9.095157e+05,AER Carbon Sequestration Agreements,3,5911050004:00,EPSG:3400,"MULTIPOLYGON (((-1157170.769 702256.75, -11575..."
4,5911050005,00,LEASE,059,PORE SPACE,ACTIVE,INITIAL TERM,SHELL CANADA LIMITED,PORE SPACE BELOW THE TOP OF THE ELK POINT GRP(...,55296.0,...,5.583449e+08,1.041488e+06,5.405159e+08,54051.588175,1.024725e+06,AER Carbon Sequestration Agreements,4,5911050005:00,EPSG:3400,"MULTIPOLYGON (((-1171999.228 719929.782, -1172..."



Columns: 24
Features: 45
CRS: EPSG:3978


In [18]:
# ---------------------------------------------------------------------------
# Build agreement-level layer
# ---------------------------------------------------------------------------

def combine_unique_text(series):
    values = (
        series
        .dropna()
        .astype(str)
        .str.strip()
    )

    values = [value for value in values if value]

    return " | ".join(dict.fromkeys(values))


agreement_gdf = (
    tract_gdf
    .dissolve(
        by="agreement_id",
        as_index=False,
        aggfunc={
            "agreement_group": "first",
            "agreement_type_code": "first",
            "mineral_type": "first",
            "status": "first",
            "vintage": "first",
            "designated_representative": "first",
            "zone_description": combine_unique_text,
            "original_area_ha": "first",
            "agreement_area_ha": "first",
            "term_date": "first",
            "continuation_date": "first",
            "current_expiry": "first",
            "source_dataset": "first",
            "source_crs": "first",
        },
    )
)

agreement_gdf["tract_count"] = (
    tract_gdf
    .groupby("agreement_id")
    .size()
    .reindex(agreement_gdf["agreement_id"])
    .to_numpy()
)

agreement_gdf["geometry_area_m2"] = agreement_gdf.geometry.area
agreement_gdf["geometry_area_ha"] = agreement_gdf["geometry_area_m2"] / 10_000
agreement_gdf["geometry_perimeter_m"] = agreement_gdf.geometry.length

print("Agreement features:", len(agreement_gdf))
print("Unique agreement IDs:", agreement_gdf["agreement_id"].nunique())
print("CRS:", agreement_gdf.crs)

display(agreement_gdf.head())

Agreement features: 43
Unique agreement IDs: 43
CRS: EPSG:3978


,agreement_id,geometry,agreement_group,agreement_type_code,mineral_type,status,vintage,designated_representative,zone_description,original_area_ha,agreement_area_ha,term_date,continuation_date,current_expiry,source_dataset,source_crs,tract_count,geometry_area_m2,geometry_area_ha,geometry_perimeter_m
0,240148301,"MULTIPOLYGON (((-1340063.13 483621.233, -13399...",APP,A59,PORE SPACE,ACTIVE,NaN,ALTAGAS LTD.,PORE SPACE IN THE WOODBEND GRP(D300),132800.0,132800.0,2024-07-02,NaT,NaT,AER Carbon Sequestration Agreements,EPSG:3400,1,1.318108e+09,131810.843867,2.523664e+06
1,250085901,"MULTIPOLYGON (((-1458051.411 913858.071, -1458...",APP,A61,PORE SPACE,ACTIVE,NaN,CANADIAN NATURAL RESOURCES LIMITED,PORE SPACE IN THE WINTERBURN GRP(D80),2304.0,2304.0,2025-05-29,NaT,NaT,AER Carbon Sequestration Agreements,EPSG:3400,1,2.234005e+07,2234.005108,4.727022e+04
2,250108401,"POLYGON ((-1250760.271 260180.227, -1251136.24...",APP,A61,PORE SPACE,ACTIVE,NaN,CENTRAL FARMS RNG LTD.,PORE SPACE IN THE LIVINGSTONE FM(D236),256.0,256.0,2025-06-30,NaT,NaT,AER Carbon Sequestration Agreements,EPSG:3400,1,2.498187e+06,249.818719,6.322875e+03
3,260043401,"MULTIPOLYGON (((-1471101.463 973653.035, -1471...",APP,A61,PORE SPACE,ACTIVE,NaN,ALTAGAS LTD.,PORE SPACE BELOW THE TOP OF THE BELLOY FM(D32)...,2240.0,2240.0,2026-03-03,NaT,NaT,AER Carbon Sequestration Agreements,EPSG:3400,1,2.175867e+07,2175.866525,4.730449e+04
4,260088401,"MULTIPOLYGON (((-1488982.91 912389.123, -14893...",APP,A61,PORE SPACE,ACTIVE,NaN,KEYERA ENERGY LTD.,PORE SPACE IN THE LEDUC FM(D67),1024.0,1024.0,2026-05-07,NaT,NaT,AER Carbon Sequestration Agreements,EPSG:3400,1,9.859792e+06,985.979189,2.512035e+04


In [19]:
# ---------------------------------------------------------------------------
# Validate agreement-level layer
# ---------------------------------------------------------------------------

print("Features:", len(agreement_gdf))
print("Unique agreement IDs:", agreement_gdf["agreement_id"].nunique())
print("Duplicate agreement IDs:", agreement_gdf["agreement_id"].duplicated().sum())

print("Null geometries:", agreement_gdf.geometry.isna().sum())
print("Empty geometries:", agreement_gdf.geometry.is_empty.sum())
print("Invalid geometries:", (~agreement_gdf.geometry.is_valid).sum())

print("\nTract counts:")
print(agreement_gdf["tract_count"].value_counts().sort_index())

print("\nGeometry types:")
print(agreement_gdf.geometry.geom_type.value_counts())

Features: 43
Unique agreement IDs: 43
Duplicate agreement IDs: 0
Null geometries: 0
Empty geometries: 0
Invalid geometries: 0

Tract counts:
tract_count
1    41
2     2
Name: count, dtype: int64

Geometry types:
MultiPolygon    41
Polygon          2
Name: count, dtype: int64


In [20]:
# ---------------------------------------------------------------------------
# Build dataset metadata table
# ---------------------------------------------------------------------------

metadata_records = [
    ("title", "Alberta Carbon Sequestration Agreements"),
    ("dataset_role", "regulatory_tenure"),
    ("assessment_type", "carbon_sequestration_agreements"),
    ("capacity_data", "False"),

    ("source_title", "Carbon Sequestration Agreements"),
    ("source_publication", "Alberta Energy Regulator"),
    ("source_year", "2026"),
    (
        "source_url",
        "https://gis.energy.gov.ab.ca/GeoView/CarbonSequestration"
    ),
    (
        "source_parent_url",
        "https://www.alberta.ca/interactive-energy-maps"
    ),

    (
        "source_documentation",
        "No standalone technical documentation was provided with the download. "
        "Available source documentation consists primarily of the ESRI shapefile XML "
        "metadata and the information pane accompanying the AER download interface."
    ),

    (
        "source_disclaimer",
        "AER states that Carbon Sequestration Agreement boundaries are shown to the "
        "full ATS Landkey and may therefore appear to include minerals not owned by "
        "the Alberta Crown and/or minerals reserved from disposition. Detailed permit "
        "and lease searches should be confirmed with Alberta Crown Land Data Support."
    ),

    ("who", "Andrew Vigars / CanCO2Re Activity 13"),

    (
        "what",
        "Harmonized Alberta carbon sequestration agreement polygons representing "
        "regulatory pore-space agreements and associated agreement attributes. "
        "The dataset does not provide quantified geological storage capacity, "
        "injectivity, storage probability, or project-ready storage resource."
    ),

    (
        "when",
        "Source shapefile exported 2026-09-08; silver-layer processing 2026-09-12."
    ),

    (
        "where",
        "Alberta, Canada; source CRS EPSG:3400; "
        "silver-layer CRS EPSG:3978."
    ),

    (
        "how",
        "Derived from the Alberta Energy Regulator Carbon Sequestration Agreements "
        "shapefile. Source fields were renamed and standardized, date fields were "
        "normalized, original geometry measurements were retained, geometries were "
        "reprojected from EPSG:3400 to EPSG:3978, geometry measurements were "
        "recalculated in the silver CRS, feature-level provenance was added, and "
        "agreement-level geometries were produced by dissolving tract features by "
        "agreement identifier. Distinct tract-level geological zone descriptions "
        "were preserved when agreements contained multiple tracts."
    ),

    (
        "credits",
        "Source dataset: Alberta Energy Regulator. "
        "Silver-layer harmonization: Andrew Vigars, CanCO2Re Activity 13."
    ),

    (
        "use_limitations",
        "Agreement polygons represent regulatory or tenure information associated "
        "with carbon sequestration pore-space rights. Boundaries are represented "
        "using the full ATS Landkey and may include minerals not owned by the Alberta "
        "Crown or minerals reserved from disposition. The polygons must not be "
        "interpreted as quantified CO2 storage capacity, injectivity, geological "
        "prospectivity, permitted injection capacity, or project-ready storage resource."
    ),

    (
        "keywords",
        "carbon storage; CO2 storage; CCUS; geological storage; pore space; "
        "carbon sequestration agreement; Alberta; Alberta Energy Regulator; CanCO2Re"
    ),

    ("source_crs", "EPSG:3400"),
    ("silver_crs", "EPSG:3978"),
    ("bronze_layer_count", "1"),
    ("canco2re_activity", "13"),
    ("creator_initials", "AV"),
]

metadata_aer_agreements = pd.DataFrame(
    metadata_records,
    columns=["key", "value"],
)

display(metadata_aer_agreements)

,key,value
0,title,Alberta Carbon Sequestration Agreements
1,dataset_role,regulatory_tenure
2,assessment_type,carbon_sequestration_agreements
3,capacity_data,False
4,source_title,Carbon Sequestration Agreements
5,source_publication,Alberta Energy Regulator
6,source_year,2026
7,source_url,https://gis.energy.gov.ab.ca/GeoView/CarbonSeq...
8,source_parent_url,https://www.alberta.ca/interactive-energy-maps
9,source_documentation,No standalone technical documentation was prov...


In [21]:
# ---------------------------------------------------------------------------
# Build QA summary table
# ---------------------------------------------------------------------------

qa_records = [
    ("source_feature_count", int(len(gdf))),
    ("tract_feature_count", int(len(tract_gdf))),
    ("agreement_feature_count", int(len(agreement_gdf))),
    (
        "unique_agreement_count",
        int(tract_gdf["agreement_id"].nunique()),
    ),
    (
        "multi_tract_agreement_count",
        int((agreement_gdf["tract_count"] > 1).sum()),
    ),

    (
        "tract_null_geometry_count",
        int(tract_gdf.geometry.isna().sum()),
    ),
    (
        "tract_empty_geometry_count",
        int(tract_gdf.geometry.is_empty.sum()),
    ),
    (
        "tract_invalid_geometry_count",
        int((~tract_gdf.geometry.is_valid).sum()),
    ),

    (
        "agreement_null_geometry_count",
        int(agreement_gdf.geometry.isna().sum()),
    ),
    (
        "agreement_empty_geometry_count",
        int(agreement_gdf.geometry.is_empty.sum()),
    ),
    (
        "agreement_invalid_geometry_count",
        int((~agreement_gdf.geometry.is_valid).sum()),
    ),

    ("source_crs", "EPSG:3400"),
    ("silver_crs", "EPSG:3978"),

    (
        "median_source_to_silver_area_ratio",
        float(measurement_check["shape_area_ratio"].median()),
    ),
    (
        "median_source_to_silver_length_ratio",
        float(measurement_check["shape_length_ratio"].median()),
    ),
]

qa_aer_agreements = pd.DataFrame(
    qa_records,
    columns=["check", "value"],
)

display(qa_aer_agreements)

,check,value
0,source_feature_count,45
1,tract_feature_count,45
2,agreement_feature_count,43
3,unique_agreement_count,43
4,multi_tract_agreement_count,2
5,tract_null_geometry_count,0
6,tract_empty_geometry_count,0
7,tract_invalid_geometry_count,0
8,agreement_null_geometry_count,0
9,agreement_empty_geometry_count,0


In [22]:
# ---------------------------------------------------------------------------
# Add standard CanCO2Re dataset classification fields
# ---------------------------------------------------------------------------

tract_gdf["assessment_type"] = "carbon_sequestration_agreement"
tract_gdf["data_class"] = "regulatory_tenure"
tract_gdf["capacity_data"] = False

agreement_gdf["assessment_type"] = "carbon_sequestration_agreement"
agreement_gdf["data_class"] = "regulatory_tenure"
agreement_gdf["capacity_data"] = False

display(
    tract_gdf[
        [
            "agreement_id",
            "tract_id",
            "assessment_type",
            "data_class",
            "capacity_data",
        ]
    ].head()
)

,agreement_id,tract_id,assessment_type,data_class,capacity_data
0,5911050001,00,carbon_sequestration_agreement,regulatory_tenure,False
1,5911050002,00,carbon_sequestration_agreement,regulatory_tenure,False
2,5911050003,00,carbon_sequestration_agreement,regulatory_tenure,False
3,5911050004,00,carbon_sequestration_agreement,regulatory_tenure,False
4,5911050005,00,carbon_sequestration_agreement,regulatory_tenure,False


In [23]:
# ---------------------------------------------------------------------------
# Build AER tract-layer field dictionary
# ---------------------------------------------------------------------------

field_dictionary_records = [
    (
        "agreement_id",
        "TEXT",
        "AER agreement identifier.",
        "identifier",
        "source harmonized",
        "Preserved as text to retain the source identifier exactly."
    ),
    (
        "tract_id",
        "TEXT",
        "AER tract identifier within an agreement.",
        "identifier",
        "source harmonized",
        "Preserved as text because source values contain leading zeros."
    ),
    (
        "agreement_group",
        "TEXT",
        "Broad AER agreement category.",
        "text",
        "source harmonized",
        "Source values include LEASE, PERMIT, and APP."
    ),
    (
        "agreement_type_code",
        "TEXT",
        "AER agreement type code.",
        "code",
        "source harmonized",
        "Source field AgreementT; codes include 058, 059, 061 and A-prefixed variants."
    ),
    (
        "mineral_type",
        "TEXT",
        "Mineral or pore-space rights type reported by AER.",
        "text",
        "source harmonized",
        "All records in the 2026 source dataset are PORE SPACE."
    ),
    (
        "status",
        "TEXT",
        "Agreement status reported by AER.",
        "text",
        "source harmonized",
        "All records in the 2026 source dataset are ACTIVE."
    ),
    (
        "vintage",
        "TEXT",
        "AER vintage or agreement-term classification.",
        "text",
        "source harmonized",
        "Source documentation does not provide a formal field definition."
    ),
    (
        "designated_representative",
        "TEXT",
        "Designated representative or agreement holder reported by AER.",
        "text",
        "source harmonized",
        "Renamed from source field DesRep."
    ),
    (
        "zone_description",
        "TEXT",
        "Geological pore-space interval or zone description associated with the tract.",
        "text",
        "source harmonized",
        "May differ between tracts belonging to the same agreement."
    ),
    (
        "original_area_ha",
        "REAL",
        "Original agreement area reported by AER.",
        "ha",
        "source harmonized",
        "Agreement-level value repeated across tracts where an agreement contains multiple tracts."
    ),
    (
        "agreement_area_ha",
        "REAL",
        "Current agreement area reported by AER.",
        "ha",
        "source harmonized",
        "Agreement-level value repeated across tracts where an agreement contains multiple tracts."
    ),
    (
        "term_date",
        "DATE",
        "Agreement term date reported by AER.",
        "date",
        "source harmonized",
        "Converted from the source text field TERMDATE."
    ),
    (
        "continuation_date",
        "DATE",
        "Agreement continuation date reported by AER.",
        "date",
        "source harmonized",
        "Source field CONTDATE; null for all features in the 2026 source dataset."
    ),
    (
        "current_expiry",
        "DATE",
        "Current agreement expiry date reported by AER.",
        "date",
        "source harmonized",
        "Converted from the source text field CUREXPIRY."
    ),
    (
        "source_shape_area_m2",
        "REAL",
        "Area measurement stored with the original AER geometry.",
        "m²",
        "source spatial",
        "Preserved from source field Shape_STAr prior to reprojection."
    ),
    (
        "source_shape_length_m",
        "REAL",
        "Boundary length measurement stored with the original AER geometry.",
        "m",
        "source spatial",
        "Preserved from source field Shape_STLe prior to reprojection."
    ),
    (
        "geometry_area_m2",
        "REAL",
        "Feature area calculated from the harmonized EPSG:3978 geometry.",
        "m²",
        "derived spatial",
        "Recalculated after reprojection."
    ),
    (
        "geometry_area_ha",
        "REAL",
        "Feature area calculated from the harmonized EPSG:3978 geometry.",
        "ha",
        "derived spatial",
        "geometry_area_m2 divided by 10,000."
    ),
    (
        "geometry_perimeter_m",
        "REAL",
        "Feature perimeter calculated from the harmonized EPSG:3978 geometry.",
        "m",
        "derived spatial",
        "Recalculated after reprojection."
    ),
    (
        "source_dataset",
        "TEXT",
        "Name of the Bronze source dataset from which the feature was derived.",
        "text",
        "provenance",
        "Set to AER Carbon Sequestration Agreements."
    ),
    (
        "source_feature_id",
        "INTEGER",
        "Sequential identifier assigned to the original source feature during harmonization.",
        "identifier",
        "provenance",
        "Corresponds to the source feature row used in this Silver-layer workflow."
    ),
    (
        "source_feature_uid",
        "TEXT",
        "Unique provenance identifier for the source tract feature.",
        "text identifier",
        "provenance",
        "Constructed as agreement_id:tract_id."
    ),
    (
        "source_crs",
        "TEXT",
        "Coordinate reference system of the original AER source geometry.",
        "CRS text",
        "provenance",
        "EPSG:3400 prior to conversion to EPSG:3978."
    ),
    (
        "assessment_type",
        "TEXT",
        "Type of storage-related assessment or dataset represented by the feature.",
        "text",
        "derived classification",
        "Set to carbon_sequestration_agreement."
    ),
    (
        "data_class",
        "TEXT",
        "High-level role of the dataset within the geological storage data model.",
        "text",
        "derived classification",
        "Set to regulatory_tenure."
    ),
    (
        "capacity_data",
        "BOOLEAN",
        "Flag indicating whether the dataset contains quantified CO2 storage capacity estimates.",
        "boolean",
        "derived classification",
        "False; this dataset contains regulatory agreement information rather than storage capacity."
    ),
    (
        "geometry",
        "POLYGON / MULTIPOLYGON",
        "Polygonal geometry representing the AER carbon sequestration agreement tract.",
        "geometry",
        "source geometry reprojected",
        "Stored in EPSG:3978."
    ),
]

aer_tract_field_dictionary = pd.DataFrame(
    field_dictionary_records,
    columns=[
        "field",
        "data_type",
        "definition",
        "units",
        "origin",
        "notes",
    ],
)

display(aer_tract_field_dictionary)

,field,data_type,definition,units,origin,notes
0,agreement_id,TEXT,AER agreement identifier.,identifier,source harmonized,Preserved as text to retain the source identif...
1,tract_id,TEXT,AER tract identifier within an agreement.,identifier,source harmonized,Preserved as text because source values contai...
2,agreement_group,TEXT,Broad AER agreement category.,text,source harmonized,"Source values include LEASE, PERMIT, and APP."
3,agreement_type_code,TEXT,AER agreement type code.,code,source harmonized,"Source field AgreementT; codes include 058, 05..."
4,mineral_type,TEXT,Mineral or pore-space rights type reported by ...,text,source harmonized,All records in the 2026 source dataset are POR...
5,status,TEXT,Agreement status reported by AER.,text,source harmonized,All records in the 2026 source dataset are ACT...
6,vintage,TEXT,AER vintage or agreement-term classification.,text,source harmonized,Source documentation does not provide a formal...
7,designated_representative,TEXT,Designated representative or agreement holder ...,text,source harmonized,Renamed from source field DesRep.
8,zone_description,TEXT,Geological pore-space interval or zone descrip...,text,source harmonized,May differ between tracts belonging to the sam...
9,original_area_ha,REAL,Original agreement area reported by AER.,ha,source harmonized,Agreement-level value repeated across tracts w...


In [24]:
# ---------------------------------------------------------------------------
# Build AER agreement-layer field dictionary
# ---------------------------------------------------------------------------

agreement_dictionary_fields = [
    "agreement_id",
    "agreement_group",
    "agreement_type_code",
    "mineral_type",
    "status",
    "vintage",
    "designated_representative",
    "zone_description",
    "original_area_ha",
    "agreement_area_ha",
    "term_date",
    "continuation_date",
    "current_expiry",
    "geometry_area_m2",
    "geometry_area_ha",
    "geometry_perimeter_m",
    "source_dataset",
    "source_crs",
    "assessment_type",
    "data_class",
    "capacity_data",
]

aer_agreement_field_dictionary = (
    aer_tract_field_dictionary[
        aer_tract_field_dictionary["field"].isin(agreement_dictionary_fields)
    ]
    .copy()
)

# Agreement-level geometry has a different interpretation than the tract layer
agreement_specific_records = pd.DataFrame(
    [
        (
            "tract_count",
            "INTEGER",
            "Number of source AER tract features represented by the dissolved agreement geometry.",
            "count",
            "derived",
            "Most agreements contain one tract; two agreements in the 2026 source dataset contain two tracts."
        ),
        (
            "geometry",
            "POLYGON / MULTIPOLYGON",
            "Polygonal geometry representing the complete AER carbon sequestration agreement.",
            "geometry",
            "derived spatial",
            "Created by dissolving tract geometries by agreement_id and stored in EPSG:3978."
        ),
    ],
    columns=[
        "field",
        "data_type",
        "definition",
        "units",
        "origin",
        "notes",
    ],
)

aer_agreement_field_dictionary = pd.concat(
    [
        aer_agreement_field_dictionary,
        agreement_specific_records,
    ],
    ignore_index=True,
)

display(aer_agreement_field_dictionary)

,field,data_type,definition,units,origin,notes
0,agreement_id,TEXT,AER agreement identifier.,identifier,source harmonized,Preserved as text to retain the source identif...
1,agreement_group,TEXT,Broad AER agreement category.,text,source harmonized,"Source values include LEASE, PERMIT, and APP."
2,agreement_type_code,TEXT,AER agreement type code.,code,source harmonized,"Source field AgreementT; codes include 058, 05..."
3,mineral_type,TEXT,Mineral or pore-space rights type reported by ...,text,source harmonized,All records in the 2026 source dataset are POR...
4,status,TEXT,Agreement status reported by AER.,text,source harmonized,All records in the 2026 source dataset are ACT...
5,vintage,TEXT,AER vintage or agreement-term classification.,text,source harmonized,Source documentation does not provide a formal...
6,designated_representative,TEXT,Designated representative or agreement holder ...,text,source harmonized,Renamed from source field DesRep.
7,zone_description,TEXT,Geological pore-space interval or zone descrip...,text,source harmonized,May differ between tracts belonging to the sam...
8,original_area_ha,REAL,Original agreement area reported by AER.,ha,source harmonized,Agreement-level value repeated across tracts w...
9,agreement_area_ha,REAL,Current agreement area reported by AER.,ha,source harmonized,Agreement-level value repeated across tracts w...


In [25]:
# ---------------------------------------------------------------------------
# Refine agreement-level field definitions
# ---------------------------------------------------------------------------

zone_mask = (
    aer_agreement_field_dictionary["field"]
    == "zone_description"
)

aer_agreement_field_dictionary.loc[
    zone_mask,
    "definition",
] = (
    "Geological pore-space interval or zone description associated "
    "with the agreement."
)

aer_agreement_field_dictionary.loc[
    zone_mask,
    "notes",
] = (
    "For single-tract agreements this preserves the source tract "
    "description. For multi-tract agreements, unique tract-level "
    "zone descriptions are concatenated using ' | ' so that no "
    "geological interval information is discarded during dissolve."
)

display(
    aer_agreement_field_dictionary[
        aer_agreement_field_dictionary["field"].isin(
            ["zone_description", "tract_count", "geometry"]
        )
    ]
)

,field,data_type,definition,units,origin,notes
7,zone_description,TEXT,Geological pore-space interval or zone descrip...,text,source harmonized,For single-tract agreements this preserves the...
21,tract_count,INTEGER,Number of source AER tract features represente...,count,derived,Most agreements contain one tract; two agreeme...
22,geometry,POLYGON / MULTIPOLYGON,Polygonal geometry representing the complete A...,geometry,derived spatial,Created by dissolving tract geometries by agre...


In [26]:
# ---------------------------------------------------------------------------
# Export field dictionaries
# ---------------------------------------------------------------------------

TRACT_DICT_CSV = (
    OUTPUT_DIR
    / "aer_agreement_tracts_field_dictionary.csv"
)

AGREEMENT_DICT_CSV = (
    OUTPUT_DIR
    / "aer_agreements_field_dictionary.csv"
)

aer_tract_field_dictionary.to_csv(
    TRACT_DICT_CSV,
    index=False,
)

aer_agreement_field_dictionary.to_csv(
    AGREEMENT_DICT_CSV,
    index=False,
)

print("Exported:")
print(TRACT_DICT_CSV)
print(AGREEMENT_DICT_CSV)

Exported:
C:\Users\aviga\Research\potential data\Storage\AER\derived\aer_agreement_tracts_field_dictionary.csv
C:\Users\aviga\Research\potential data\Storage\AER\derived\aer_agreements_field_dictionary.csv


In [27]:
# ---------------------------------------------------------------------------
# Export finalized AER Silver GeoPackage
# ---------------------------------------------------------------------------

OUTPUT_GPKG = (
    OUTPUT_DIR
    / "20260912_13_AERCarbonSequestrationAgreements_AV.gpkg"
)

# Replace any previous exploratory output
if OUTPUT_GPKG.exists():
    OUTPUT_GPKG.unlink()

# ---------------------------------------------------------------------------
# Write spatial layers
# ---------------------------------------------------------------------------

tract_gdf.to_file(
    OUTPUT_GPKG,
    layer="aer_agreement_tracts",
    driver="GPKG",
)

agreement_gdf.to_file(
    OUTPUT_GPKG,
    layer="aer_agreements",
    driver="GPKG",
)

# ---------------------------------------------------------------------------
# Write non-spatial metadata and QA tables
# ---------------------------------------------------------------------------

with sqlite3.connect(OUTPUT_GPKG) as conn:

    metadata_aer_agreements.to_sql(
        "metadata_aer_agreements",
        conn,
        if_exists="replace",
        index=False,
    )

    qa_aer_agreements.to_sql(
        "qa_aer_agreements",
        conn,
        if_exists="replace",
        index=False,
    )

    # Register non-spatial tables as GeoPackage attribute tables
    conn.execute(
        """
        INSERT OR REPLACE INTO gpkg_contents (
            table_name,
            data_type,
            identifier,
            description
        )
        VALUES (?, 'attributes', ?, ?)
        """,
        (
            "metadata_aer_agreements",
            "metadata_aer_agreements",
            "Dataset-level provenance, processing, and use metadata.",
        ),
    )

    conn.execute(
        """
        INSERT OR REPLACE INTO gpkg_contents (
            table_name,
            data_type,
            identifier,
            description
        )
        VALUES (?, 'attributes', ?, ?)
        """,
        (
            "qa_aer_agreements",
            "qa_aer_agreements",
            "Dataset-level quality-assurance and validation summary.",
        ),
    )

print("Exported GeoPackage:")
print(OUTPUT_GPKG)

Exported GeoPackage:
C:\Users\aviga\Research\potential data\Storage\AER\derived\20260912_13_AERCarbonSequestrationAgreements_AV.gpkg


In [28]:
# ---------------------------------------------------------------------------
# Validate spatial layers
# ---------------------------------------------------------------------------

print("GeoPackage exists:", OUTPUT_GPKG.exists())

tract_check = gpd.read_file(
    OUTPUT_GPKG,
    layer="aer_agreement_tracts",
)

agreement_check = gpd.read_file(
    OUTPUT_GPKG,
    layer="aer_agreements",
)

print("\nTract layer:")
print("Features:", len(tract_check))
print("CRS:", tract_check.crs)
print("Invalid geometries:", (~tract_check.geometry.is_valid).sum())

print("\nAgreement layer:")
print("Features:", len(agreement_check))
print("CRS:", agreement_check.crs)
print("Invalid geometries:", (~agreement_check.geometry.is_valid).sum())

# ---------------------------------------------------------------------------
# Validate registered GeoPackage tables
# ---------------------------------------------------------------------------

with sqlite3.connect(OUTPUT_GPKG) as conn:

    contents = pd.read_sql_query(
        """
        SELECT
            table_name,
            data_type,
            identifier,
            description
        FROM gpkg_contents
        ORDER BY table_name
        """,
        conn,
    )

    metadata_check = pd.read_sql_query(
        "SELECT * FROM metadata_aer_agreements",
        conn,
    )

    qa_check = pd.read_sql_query(
        "SELECT * FROM qa_aer_agreements",
        conn,
    )

display(contents)

print("\nMetadata rows:", len(metadata_check))
print("QA rows:", len(qa_check))

display(qa_check)

GeoPackage exists: True

Tract layer:
Features: 45
CRS: EPSG:3978
Invalid geometries: 0

Agreement layer:
Features: 43
CRS: EPSG:3978
Invalid geometries: 0


,table_name,data_type,identifier,description
0,aer_agreement_tracts,features,aer_agreement_tracts,
1,aer_agreements,features,aer_agreements,
2,metadata_aer_agreements,attributes,metadata_aer_agreements,"Dataset-level provenance, processing, and use ..."
3,qa_aer_agreements,attributes,qa_aer_agreements,Dataset-level quality-assurance and validation...



Metadata rows: 24
QA rows: 15


,check,value
0,source_feature_count,45
1,tract_feature_count,45
2,agreement_feature_count,43
3,unique_agreement_count,43
4,multi_tract_agreement_count,2
5,tract_null_geometry_count,0
6,tract_empty_geometry_count,0
7,tract_invalid_geometry_count,0
8,agreement_null_geometry_count,0
9,agreement_empty_geometry_count,0


In [29]:
# ---------------------------------------------------------------------------
# Write GeoPackage auxiliary metadata
# ---------------------------------------------------------------------------

AUX_XML = OUTPUT_GPKG.with_name(
    OUTPUT_GPKG.name + ".aux.xml"
)

aux_xml = """<PAMDataset>
  <Metadata>
    <MDI key="CREATOR">Andrew Vigars / CanCO2Re Activity 13</MDI>
    <MDI key="CRS">EPSG:3978</MDI>
    <MDI key="DATE">2026-09-12</MDI>
    <MDI key="SOURCE">Alberta Energy Regulator Carbon Sequestration Agreements</MDI>
    <MDI key="TITLE">Alberta Carbon Sequestration Agreements</MDI>
  </Metadata>
</PAMDataset>
"""

AUX_XML.write_text(
    aux_xml,
    encoding="utf-8",
)

print("Exported auxiliary metadata:")
print(AUX_XML)

print("\nContents:")
print(AUX_XML.read_text(encoding="utf-8"))

Exported auxiliary metadata:
C:\Users\aviga\Research\potential data\Storage\AER\derived\20260912_13_AERCarbonSequestrationAgreements_AV.gpkg.aux.xml

Contents:
<PAMDataset>
  <Metadata>
    <MDI key="CREATOR">Andrew Vigars / CanCO2Re Activity 13</MDI>
    <MDI key="CRS">EPSG:3978</MDI>
    <MDI key="DATE">2026-09-12</MDI>
    <MDI key="SOURCE">Alberta Energy Regulator Carbon Sequestration Agreements</MDI>
    <MDI key="TITLE">Alberta Carbon Sequestration Agreements</MDI>
  </Metadata>
</PAMDataset>

